In [ ]:
import pickle

import pandas as pd
from torch import layout

from adaptive_latents import ArrayWithTime
from adaptive_latents import CONFIG
import pandas
from adaptive_latents.utils import angle_between
import matplotlib.pyplot as plt
import numpy as np
# import seaborn as sns

In [ ]:
def srs_to_l_df(srs):
    records = []
    for k, sr_list in srs.items():
        for sr_i, sr in enumerate(sr_list):
            latents: ArrayWithTime = sr.log['latents']
            for l_i, l in enumerate(sr.stim_designer.log):
                t_of_stim = l['time_of_stim']
                stim_sample = latents.time_to_sample(t_of_stim)
                old_v = latents[stim_sample-1] - latents[stim_sample-2]
                this_v = latents[stim_sample] - latents[stim_sample-1]
                l['old_v'] = old_v
                l['this_v'] = this_v

                records.append(dict(sr_key=k, sr_i=sr_i, l_i=l_i, l=l))
    return pandas.DataFrame(records)


import sys
sys.path.append("/home/jgould/Documents/2026_paper/code/")
with open("/mnt/data/gould_2026_cache/optim_open_vs_closed_077287296748976.pickle", 'rb') as f:
    data = pickle.load(f)
    l_df = srs_to_l_df(data)


In [ ]:
print(l_df.l[0].keys())
print(l_df.columns)

In [ ]:
l_df['sr_key'].str.split(pat=' ', expand=True)

In [ ]:
# l = df.l[0]
# np.sign(np.linalg.det(np.squeeze([l['v'][:,0], l['s'], l['this_v']])))

In [ ]:
l_df['theta'] = l_df['l'].apply(lambda l: angle_between(l['v'], l['observed_s_hat'], radians=False))
l_df['r'] = l_df['l'].apply(lambda l: np.linalg.norm(l['observed_s_hat']))

# df['theta'] = df['l'].apply(lambda l: angle_between(l['v'], l['s'], radians=False) )
# df['r'] = df['l'].apply(lambda l: np.linalg.norm(l['s']))


l_df['t'] = l_df['l'].apply(lambda l: l['time_of_stim'])
l_df['norm_u'] = l_df['l'].apply(lambda l: np.linalg.norm(l['u']))


In [ ]:
%matplotlib qt
fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, layout='constrained')

min_norm = 7.5
max_angle = 15

for k, ax in zip(l_df.sr_key.unique(), axs.flatten()):
    sub_df = l_df[l_df.sr_key == k]
    sub_df = sub_df[sub_df.t > sub_df.t.median()]
    ax.scatter(sub_df['theta'], sub_df['r'], s=1, label=k, c=sub_df.sr_i)
    patch = plt.Rectangle(xy=(0,min_norm), width=max_angle, height=100, color='#777777', alpha=.1)
    ax.add_patch(patch)
    ax.set_xlim(xmin=0, xmax=180)
    ax.text(0.99, 0.97, k + f"\n {((sub_df.theta < max_angle) & (sub_df.r > min_norm)).sum()} / {len(sub_df)}", transform=ax.transAxes, ha='right', va='top')

    # ax.legend()
    ax.set_ylim(0,60)
    ax.axvline(90, linestyle='--', color='gray', alpha=0.5)

for ax in axs[-1,:]:
    ax.set_xlabel('Angle between v and $s_{\\text{obs}}$ (degrees)')

for ax in axs[:,0]:
    ax.set_ylabel('Norm of $s_{\\text{obs}}$')

In [ ]:
l_df.sr_key.unique()

In [ ]:
%matplotlib inline


for k in l_df.sr_key.unique():
    fig, ax = plt.subplots(figsize=(4,4), layout='constrained')

    sub_df = l_df[l_df.sr_key == k]
    sub_df = sub_df[sub_df.t > sub_df.t.median()]
    ax.scatter(sub_df['r'], sub_df['theta'],  s=5, label=k, color='k')
    ax.set_ylim(ymin=0, ymax=180)
    ax.set_xlim(0,60)
    ax.axhline(90, linestyle='--', color='gray', alpha=0.5)

    ax.set_ylabel('Alignment error (angle in degrees)', fontsize=12)
    ax.set_xlabel('Response magnitude', fontsize=12)
    fig.savefig(CONFIG.plot_save_path / f'{k}.svg', transparent=True)

# patch = plt.Rectangle(xy=(0,min_norm), width=max_angle, height=100, color='#777777', alpha=.1)
# ax.add_patch(patch)
# ax.text(0.99, 0.98,  f"{((sub_df.theta < max_angle) & (sub_df.r > min_norm)).sum()} / {len(sub_df)}", transform=ax.transAxes, ha='right', va='top')



In [ ]:

fig, ax = plt.subplots(figsize=(6,6), layout='constrained')

sub_df = l_df[l_df.sr_key == k]
sub_df = sub_df[sub_df.t > sub_df.t.median()]
ax.scatter(sub_df['theta'], sub_df['r'], s=5, label=k, c=sub_df['norm_u'])
ax.set_xlim(xmin=0, xmax=180)
ax.set_ylim(0,60)
ax.axvline(90, linestyle='--', color='gray', alpha=0.5)

ax.set_xlabel('Angle between v and $s_{\\text{obs}}$ (degrees)')
ax.set_ylabel('Norm of $s_{\\text{obs}}$')


In [ ]:
sub_df.sort_values(by='norm_u').iloc[0,3]['u']

In [ ]:
l_df.sr_key.unique()

In [ ]:

# sub_df = l_df[l_df.sr_key == k]
sub_df = l_df[l_df.sr_key.apply(lambda x: 'closed' in x)]

fig, ax = plt.subplots()
for k,v in sub_df.groupby(by=['sr_i', 'l_i']):
    plt.plot(v[v.sr_key == 'closed id'].r, v[v.sr_key == 'closed flip'].r, '.')


In [ ]:
v

In [ ]:
%matplotlib inline
fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, subplot_kw=dict(projection='polar'), layout='constrained')

for k, ax in zip(l_df.sr_key.unique(), axs.flatten()):
    sub_df = l_df[l_df.sr_key == k]
    sub_df = sub_df[sub_df.t > sub_df.t.median()]
    ax.scatter(sub_df['theta'] * np.pi/180, sub_df['r'], c=sub_df['t'], s=1, label=k)

    ax.set_ylim(0,40)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    patch = plt.Rectangle(xy=(0,min_norm), width=max_angle * np.pi/180, height=100, color='r', alpha=.1)
    ax.add_patch(patch)


    ax.text(0.99, 0.97, k + f"\n {((sub_df.theta < max_angle) & (sub_df.r > min_norm)).sum()} / {len(sub_df)}", transform=ax.transAxes, ha='right', va='top')


In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, layout='constrained')

for k, ax in zip(l_df.sr_key.unique(), axs.flatten()):
    ax = axs[0,0]
    sub_df = l_df[(l_df.sr_key == k) & (l_df.r > 5)]
    sns.regplot(sub_df, x='t', y='theta', ax=ax, scatter_kws={'s':0}, line_kws={'color':'red'}, lowess=True)
    # ax.scatter(sub_df['t'], sub_df['theta'], s=1, label=k, color='k')
